# Databricks 版 LangChain 实验（对标 IBM 教程）

这个 notebook 按照 IBM 教程的学习路径构建：
1. Prompt Templates
2. Chains（LCEL）
3. RAG（检索增强生成）
4. Tools and Agents

在 Databricks 上建议先准备：
- 一个可用的 Foundation Model Serving Endpoint
- （可选）Embedding Endpoint
- （可选）Databricks Vector Search Index

In [ ]:
# 在 Databricks notebook 中运行
%pip install -U langchain langchain-core langchain-community databricks-langchain mlflow databricks-vectorsearch

In [ ]:
from databricks_langchain import ChatDatabricks
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO: 改成你自己的 Databricks Serving Endpoint 名称
CHAT_ENDPOINT = "databricks-meta-llama-3-1-70b-instruct"

llm = ChatDatabricks(endpoint=CHAT_ENDPOINT, temperature=0.2, max_tokens=512)
llm.invoke("用一句话介绍 LangChain")

In [ ]:
# 1) Prompt Template + Chain (LCEL)
prompt = PromptTemplate.from_template(
    "你是一个数据平台专家。请用{style}风格解释：{topic}，并给出3条建议。"
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({
    "style": "简洁",
    "topic": "为什么在企业里使用 RAG"
})
print(result)

In [ ]:
# 2) RAG 最小可运行版（先用内存文档模拟）
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

raw_docs = [
    Document(page_content="LangChain provides prompt templates, chains, memory, and agent abstractions."),
    Document(page_content="RAG combines retrieval and generation to reduce hallucinations and improve factuality."),
    Document(page_content="Databricks can host models and support production ML/LLM workloads with governance."),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_documents(raw_docs)

emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = FAISS.from_documents(chunks, emb)
retriever = vs.as_retriever(search_kwargs={"k": 2})

question = "为什么 RAG 能提升回答准确性？"
ctx_docs = retriever.invoke(question)
context = "\n\n".join([d.page_content for d in ctx_docs])

rag_prompt = ChatPromptTemplate.from_template(
    "基于以下上下文回答问题。如果上下文没有答案，请明确说明。\n\n上下文:\n{context}\n\n问题:\n{question}"
)

rag_chain = rag_prompt | llm | StrOutputParser()
print(rag_chain.invoke({"context": context, "question": question}))

In [ ]:
# 3) Tools + Agent（对标 IBM 教程 Agent 章节）
from langchain_core.tools import Tool
from langchain.agents import create_react_agent, AgentExecutor


def calculator(expr: str) -> str:
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"calc error: {e}"


def text_format(input_text: str) -> str:
    # 格式: uppercase: hello world
    if ":" not in input_text:
        return "格式错误，示例: uppercase: hello world"
    mode, text = input_text.split(":", 1)
    mode = mode.strip().lower()
    text = text.strip()
    if mode == "uppercase":
        return text.upper()
    if mode == "lowercase":
        return text.lower()
    if mode == "titlecase":
        return text.title()
    return "只支持 uppercase/lowercase/titlecase"


tools = [
    Tool(name="calculator", func=calculator, description="执行基础数学表达式"),
    Tool(name="text_format", func=text_format, description="格式化文本大小写"),
]

agent_prompt = PromptTemplate.from_template(
    """你是一个会调用工具的助手。

可用工具:
{tools}

工具名列表: {tool_names}

请严格使用以下格式:
Question: 用户问题
Thought: 你的思考
Action: 工具名（必须来自[{tool_names}]）
Action Input: 传给工具的输入
Observation: 工具返回
Thought: 我已经知道答案
Final Answer: 给用户的最终答案

Question: {input}
{agent_scratchpad}
"""
)

agent = create_react_agent(llm=llm, tools=tools, prompt=agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

print(agent_executor.invoke({"input": "请计算 125 * 16"})["output"])
print(agent_executor.invoke({"input": "把 hello databricks 转成 titlecase"})["output"])

## 可选：升级为 Databricks 生产版 RAG

上面的 RAG 是本地向量库演示。迁移到 Databricks 生产版时，建议替换为：
- Embedding: Databricks Embedding Serving Endpoint
- Retrieval: Databricks Vector Search Index
- Tracking: MLflow tracing + Unity Catalog 管理数据/模型/权限

如果你要，我可以下一步直接把这一格改成 Databricks Vector Search 的可执行模板（只留 endpoint/index 名称给你填）。